# 5. Single Shot 0 and 1 Measurements

In [ ]:
print('Measure lenght', qubit_parameters["ro_len"])
print('Aq port delay', lsg_q0["acquire_line"].port_delay)

In [ ]:
#lsg_q0["acquire_line"].port_delay = 0.5e-07

In [ ]:
ro_len_test = 1e-6
ro_amp_test = 0.55

In [ ]:
readout_low['readout_pulse']

In [ ]:
readout_pulse_test = pulse_library.const(
    uid="readout_pulse",
    length=ro_len_test,
    amplitude=ro_amp_test,
)

readout_weighting_function_test = pulse_library.const(
            uid="readout_weighting_function", length=ro_len_test, amplitude=1.0
        )

readout_test = {'readout_type': 'pulses',
                'readout_pulse': readout_pulse_test,
                'readout_weighting_function': readout_weighting_function_test,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq_opt"],
                'acquire_freq': qubit_parameters["ro_freq_opt"],
                'readout_range': -20,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

## 5.1 One Single Shot Measurement

In [ ]:
# qubit_parameters.update_parameter("relax", 100e-3)
# readout_long_wait = readout_low.copy()
readout_long_wait['relax_time'] = 500e-3

In [ ]:
# how many averages per point: 2^n_average
n_average = 17

#make two-point sweep
sweep_2 = make_two_point_sweep()

exp_rabi_SS = create_rabi_SS(sweep_2, 
               x180, 
               readout_opt,#readout_low,
               n_average)

# compile
exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)

# show_pulse_sheet("Single Shot Test", exp_rabi_SS_comp)

In [ ]:
#run the experiment
SS_results = my_session.run(exp_rabi_SS_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
SS_res = SS_results.get_data("SS_rabi")

# define amplitude axis from qubit parameters
SS_it = SS_results.get_axis("SS_rabi")[0]
SS_amp = SS_results.get_axis("SS_rabi")[1]

In [ ]:
Data_SS = {'rabi_SS_res': SS_res,
           'rabi_SS_amp': SS_amp,
           'rabi_SS_it': SS_it
            }

Data_SS.update(qubit_parameters._params)
Data_SS.update(lo_settings._params)

file_name = 'Single_shots_all_unsh_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_SS)

In [ ]:
# how many averages per point: 2^n_average
n_average = 17

states = ['g', 'e', 'f']

exp_SS_comp_list = []

for state in states:
    exp_SS = create_SS(state, 
                            x180, 
                            x180_ef,
                            readout_opt,#readout_opt,#readout_low,
                            n_average)

    exp_SS_comp_list.append(my_session.compile(exp_SS))

In [ ]:
SS_results_states = []

for exp_comp in exp_SS_comp_list:
    SS_results_states.append(my_session.run(exp_comp))  

In [ ]:
result_states = []

for result in SS_results_states:
    result_states.append(result.get_data("shots"))

In [ ]:
result_arr = np.array(result_states)
result_arr.shape

In [ ]:
#for old way

# zero_data = SS_res[:,0]
# one_data = SS_res[:,1]

#for new way

zero_data = result_arr[0,:]
one_data = result_arr[1,:]
two_data = result_arr[2,:]

In [ ]:
# zero_data = SS_res[:,0]
# one_data = SS_res[:,1]

# plot measurement data on IQ
fig = plt.figure()
#plt.plot(rabi_amp, abs(rabi_res), ".k")
plt.plot(zero_data.real, zero_data.imag, '.', alpha=0.1)
#plt.plot(rabi_SS_res[:,15].real, rabi_SS_res[:,15].imag, '.')
plt.plot(one_data.real, one_data.imag, '.', alpha=0.1)
plt.plot(two_data.real, two_data.imag, '.', alpha=0.01)
plt.ylabel("Imaginary part")
plt.xlabel("Real part")

plt.xlim(-0.1, 0.12)
plt.ylim(-0.25, 0.05)

plt.plot(np.mean(zero_data.real), np.mean(zero_data.imag), 'bo')
plt.plot(np.mean(one_data.real), np.mean(one_data.imag), 'ro')
plt.plot(np.mean(two_data.real), np.mean(two_data.imag), 'yo')

# figname = 'Single_shot_points_'
# file_path = get_path_to_file(figname, '.png')
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

In [ ]:
def analize_Single_Shots(SS_results, plot = False, n_bins = 50):
    SS_res = SS_results.get_data("SS_rabi")

    print(SS_res)

    Data_0 = SS_res[:,0]
    Data_1 = SS_res[:,1]

    zero_mean = np.mean(Data_0)
    one_mean = np.mean(Data_1)

    mid_point = 0.5*(zero_mean+one_mean)

    Data_0_corr = Data_0 - mid_point
    Data_1_corr = Data_1 - mid_point

    zero_mean_corr = zero_mean - mid_point

    r = np.abs(zero_mean-one_mean)/2

    phi = np.angle(zero_mean_corr)

    Data_0_rot = Data_0_corr*np.exp(1j*(np.pi-phi))
    Data_1_rot = Data_1_corr*np.exp(1j*(np.pi-phi))

    M_z = np.mean(Data_0_rot)
    M_o = np.mean(Data_1_rot)

    STD_x_0 = np.sqrt(np.var(Data_0_rot.real, ddof=1))
    STD_y_0 = np.sqrt(np.var(Data_0_rot.imag, ddof=1))
    
    STD_x_1 = np.sqrt(np.var(Data_1_rot.real, ddof=1))
    STD_y_1 = np.sqrt(np.var(Data_1_rot.imag, ddof=1))

    print('Number of posive elements:', np.sum(np.array(Data_0_rot.real) > 0))
    print('Number of negative elements:', np.sum(np.array(Data_0_rot.real) <= 0))

    Results = {'distance': r,
               'mean_0': M_z,
               'mean_1': M_o,
               'STD_x_0': STD_x_0,
               'STD_y_0': STD_y_0,
               'STD_x_1': STD_x_1,
               'STD_y_1': STD_y_1,
               'Rel_STD_0': STD_x_0/r,
               'Rel_STD_1': STD_x_1/r
              }
    if plot:
        n_bins = 50

        fig, axs = plt.subplots(1, 2, sharey=True, tight_layout=True)

        # We can set the number of bins with the *bins* keyword argument.
        axs[0].title.set_text('Real')
        axs[0].hist(Data_0_rot.real, bins=n_bins, alpha=0.5)
        axs[0].hist(Data_1_rot.real, bins=n_bins, alpha=0.5)
        axs[0].set_yscale('log')

        axs[1].title.set_text('Imag')
        axs[1].hist(Data_0_rot.imag, bins=n_bins, alpha=0.5)
        axs[1].hist(Data_1_rot.imag, bins=n_bins, alpha=0.5)
        axs[1].set_yscale('log')
    
    return Results

In [ ]:
res = analize_Single_Shots(SS_results, plot = True)

In [ ]:
res2 = analize_Single_Shots(SS_results, plot = True)

In [ ]:
res

In [ ]:
#plot distributions

n_bins = 50

fig, axs = plt.subplots(1, 2, sharey=True, tight_layout=True)

# We can set the number of bins with the *bins* keyword argument.
axs[0].title.set_text('Real')
axs[0].hist(zero_data.real, bins=n_bins)
axs[0].hist(one_data.real, bins=n_bins)

axs[1].title.set_text('Imag')
axs[1].hist(zero_data.imag, bins=n_bins)
axs[1].hist(one_data.imag, bins=n_bins)

# figname = 'Single_shot_hist_'
# file_path = get_path_to_file(figname, '.png')
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

In [ ]:
n_bins =200
plt.hist(zero_data_proj/r, bins=n_bins, alpha=0.5)
plt.hist(one_data_proj/r, bins=n_bins, alpha=0.5)
plt.axvline(1)
plt.axvline(-1)
plt.yscale('log')
plt.ylabel('N points')
plt.xlabel('Relative distance')

In [ ]:
data = zero_data_proj/r

n, bins, patches = plt.hist(data, bins=n_bins, alpha=0.5, density=True)

(mu, sigma) = norm.fit(data)

# add a 'best fit' line
y = norm.pdf(bins, mu, sigma)
l = plt.plot(bins, y, 'r--', linewidth=2)
plt.yscale('log')

In [ ]:
data = one_data_proj/r

y, x, patches = plt.hist(data, bins=n_bins, color='red', alpha=0.25)
x=(x[1:]+x[:-1])/2

expected = (-1, .2, 1050, 1, .2, 125)
params, cov = curve_fit(bimodal, x, y, expected)
sigma=np.sqrt(np.diag(cov))
x_fit = np.linspace(x.min(), x.max(), 500)
#plot combined...
plt.plot(x_fit, bimodal(x_fit, *params), color='green', lw=3, label='One+Zero')
#...and individual Gauss curves
plt.plot(x_fit, gauss(x_fit, *params[:3]), color='red', lw=2, ls="--", label='One')
plt.plot(x_fit, gauss(x_fit, *params[3:]), color='b', lw=2, ls=":", label='Zero')
#and the original data points if no histogram has been created before
#plt.scatter(x, y, marker="X", color="black", label="original data")
plt.yscale('log')
plt.ylabel('N points')
plt.xlabel('Relative distance')
plt.ylim(1e-1, 5e3)
plt.legend()
print(pd.DataFrame(data={'params': params, 'sigma': sigma}, index=bimodal.__code__.co_varnames[1:]))
plt.show() 

print('Area ratio:', params[4]*params[5]/(params[1]*params[2]))

In [ ]:
data = zero_data_proj/r

y, x, patches = plt.hist(data, bins=n_bins, color='blue', alpha=0.25)
x=(x[1:]+x[:-1])/2

expected = (-1.5, 1.0, 250, 1, 1.0, 2025)
params, cov = curve_fit(bimodal, x, y, expected)
sigma=np.sqrt(np.diag(cov))
x_fit = np.linspace(x.min(), x.max(), 500)
#plot combined...
plt.plot(x_fit, bimodal(x_fit, *params), color='green', lw=3, label='One+Zero')
#...and individual Gauss curves
plt.plot(x_fit, gauss(x_fit, *params[:3]), color='red', lw=2, ls="--", label='One')
plt.plot(x_fit, gauss(x_fit, *params[3:]), color='blue', lw=2, ls=":", label='Zero')
#and the original data points if no histogram has been created before
#plt.scatter(x, y, marker="X", color="black", label="original data")
plt.yscale('log')
plt.ylabel('N points')
plt.xlabel('Relative distance')
plt.ylim(1e-1, 5e3)
plt.legend()
print(pd.DataFrame(data={'params': params, 'sigma': sigma}, index=bimodal.__code__.co_varnames[1:]))
plt.show() 

print('Area ratio:', params[1]*params[2]/(params[4]*params[5]))

In [ ]:
compute_pca(zero_data.real, zero_data.imag, plot=True, scaling_fac=0.1)

pca_ss_dict = compute_pca(one_data.real, one_data.imag, plot=True, scaling_fac=0.1)

In [ ]:
pca_ss_dict

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [ ]:
n_bins = 50

fig, ax = plt.subplots(tight_layout=True)
hist = ax.hist2d(zero_data.real, zero_data.imag, n_bins)
hist = ax.hist2d(one_data.real, one_data.imag, n_bins)

## 5.2 Single Shot Measurements for Different Rabi Frequencies

In [ ]:
#Sweep for different pi-pulses
n_average = 17

rabi_SS_sweep_list = []

pulse_length = np.linspace(60e-9, 400e-9, 36)
rabi_freq = 1/pulse_length
pi_amp = qubit_parameters['rabi_slope']*rabi_freq*1e-6+qubit_parameters['rabi_intercept']

for i in range(len(pulse_length)):
    
    sweep_2 = make_two_point_sweep(stop=pi_amp[i])

    gaussian_pulse = pulse_library.gaussian(
        uid="gaussian_pulse", length=pulse_length[i], amplitude=1.0
    )
    
    exp_rabi_SS = make_rabi_SS(sweep_2, 
                     gaussian_pulse, 
                     readout_pulse, 
                     readout_weighting_function, 
                     qubit_parameters["relax"], 
                     n_average)
    
    exp_rabi_SS.set_signal_map(qubit_meas_map)
    exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)
    results_SS = my_session.run(exp_rabi_SS_comp)
    
    rabi_SS_sweep_list.append(results_SS.get_data("SS_rabi"))
    
rabi_SS_sweep_arr = np.array(rabi_SS_sweep_list)

In [ ]:
pi_amp

In [ ]:
rabi_freq*1e-6

In [ ]:
Data_SS = {'rabi_SS_sweep_arr': rabi_SS_sweep_arr,
           'rabi_SS_amp': SS_amp,
           'rabi_SS_it': SS_it,
           'pulse_length': pulse_length,
           'rabi_freq': rabi_freq,
           'pi_amp': pi_amp
            }

Data_SS.update(qubit_parameters)
Data_SS.update(lo_settings)

file_name = 'Single_shots_0_pi_diff_pipulses_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_SS)

## 5.3 Single Shot Measurement for Different Aquisition Lengths and Delays

In [ ]:
print('Measure lenght', readout_pulse.length)
print('Aq port delay', lsg_q0["acquire_line"].port_delay)

Here the readout pulse amplitude and length are fixed and we vary the delay and window length

In [ ]:
#Sweep for different readout length
n_average = 17

rabi_SS_sweep_list = []

#ro_len = np.linspace(100, 2000, 20)*1e-9

port_delay_sw = np.linspace(380, 400, 5)*1e-9

for k in range(len(port_delay_sw)):
    print('Delay: ', port_delay_sw[k]) 
    
    rabi_SS_sweep_list_int = []
    
    lsg_q0["acquire_line"].port_delay = port_delay_sw[k]
    
    #ro_len = np.arange(100, 2100 - port_delay_sw[k], 100)*1e-9
    ro_len = [readout_pulse.length]

    for i in range(len(ro_len)):

        print('Ro_len: ', ro_len[i])
    
        readout_weighting_function_i = pulse_library.const(
            uid="readout_weighting_function", length=ro_len[i], amplitude=1.0
        )

        exp_rabi_SS = make_rabi_SS(sweep_2, 
                         x180, 
                         readout_pulse, 
                         readout_weighting_function_i, 
                         qubit_parameters["relax"], 
                         n_average)
    
        exp_rabi_SS.set_signal_map(qubit_meas_map)
        exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)
        results_SS = my_session.run(exp_rabi_SS_comp)

        res = analize_Single_Shots(results_SS, plot = False)
    
        rabi_SS_sweep_list_int.append(res)
    
    rabi_SS_sweep_list.append(rabi_SS_sweep_list_int)
    

In [ ]:
readout_pulse.amplitude

In [ ]:
ro_len

In [ ]:
min_val = []
argmin_val = []

for i in range(len(rabi_SS_sweep_list)):
    l = [d['Rel_STD_0'] for d in rabi_SS_sweep_list[i]]
    min_val.append(np.min(np.array(l)))
    argmin_val.append(np.argmin(np.array(l)))

MM = np.argmin(np.array(min_val))
#ro_len = np.arange(100, 2100 - port_delay_sw[MM], 100)*1e-9
ro_len = [readout_pulse.length]

print('Readout length is', readout_pulse.length)
print('Minimal relative error value is ', np.round(min_val[MM], 3), 'for delay', np.round(port_delay_sw[MM]*1e9,3), 'ns (index', MM, ') and length', ro_len[argmin_val[MM]], '(index', argmin_val[MM], ').')
print('Optimal port delay is between ', np.round(port_delay_sw[MM-1]*1e9,3), ' and ', np.round(port_delay_sw[MM+1]*1e9,3), 'ns')

In [ ]:
print('Optimal port delay is between ', np.round(port_delay_sw[1]*1e9,3), ' and ', np.round(port_delay_sw[2]*1e9,3), 'ns')

In [ ]:
lsg_q0["acquire_line"].port_delay = 3.8e-7

In [ ]:
#Sweep power and length of the readout pulse. Here delay is fixed and window is equal to pulse length

#Sweep for different readout length
n_average = 17

rabi_SS_sweep_list = []

#ro_len = np.linspace(100, 2000, 20)*1e-9

ro_amp_sw = np.linspace(0.05, 0.8, 21)
ro_len_sw = np.linspace(100, 2000, 10)*1e-9

for k in range(len(ro_amp_sw)):
    print('RO Amplitude: ', ro_amp_sw[k]) 
    
    rabi_SS_sweep_list_int = []
    
    readout_pulse.amplitude = ro_amp_sw[k]

    for i in range(len(ro_len_sw)):

        print('Ro_len: ', ro_len_sw[i])

        readout_pulse.length = ro_len_sw[i]
    
        readout_weighting_function_i = pulse_library.const(
            uid="readout_weighting_function", length=ro_len_sw[i], amplitude=1.0
        )

        exp_rabi_SS = make_rabi_SS(sweep_2, 
                         x180, 
                         readout_pulse, 
                         readout_weighting_function_i, 
                         qubit_parameters["relax"], 
                         n_average)
    
        exp_rabi_SS.set_signal_map(qubit_meas_map)
        exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)
        results_SS = my_session.run(exp_rabi_SS_comp)

        res = analize_Single_Shots(results_SS, plot = False)
    
        rabi_SS_sweep_list_int.append(res)
    
    rabi_SS_sweep_list.append(rabi_SS_sweep_list_int)

In [ ]:
min_val = []
argmin_val = []

for i in range(len(rabi_SS_sweep_list)):
    l = [d['Rel_STD_0'] for d in rabi_SS_sweep_list[i]]
    min_val.append(np.min(np.array(l)))
    argmin_val.append(np.argmin(np.array(l)))

MM = np.argmin(np.array(min_val))
#ro_len = np.arange(100, 2100 - port_delay_sw[MM], 100)*1e-9
#ro_len = [readout_pulse.length]

#print('Readout length is', readout_pulse.length)
print('Minimal relative error value is ', np.round(min_val[MM], 3), 'for amplitude', np.round(ro_amp_sw[MM],3), ' (index', MM, ') and length', ro_len_sw[argmin_val[MM]], '(index', argmin_val[MM], ').')
#print('Optimal port delay is between ', np.round(port_delay_sw[MM-1]*1e9,3), ' and ', np.round(port_delay_sw[MM+1]*1e9,3), 'ns')

In [ ]:
for i in range(len(rabi_SS_sweep_list)):
    l = [d['Rel_STD_0'] for d in rabi_SS_sweep_list[:][i]]
    plt.plot(ro_amp_sw, np.array(i))

In [ ]:
STD_list = []
for l in rabi_SS_sweep_list:
    STD_list.append([d['Rel_STD_0'] for d in l])

STD_array = np.array(STD_list)

for k in range(len(ro_len_sw)):
    plt.plot(ro_amp_sw, STD_array[:,k])

plt.yscale('log')

In [ ]:
len(rabi_SS_sweep_list[6])

In [ ]:
[d['Rel_STD_0'] for d in rabi_SS_sweep_list[17]]

In [ ]:
# get measurement data returned by the instruments
rabi_SS_res = rabi_SS_sweep_arr[19,19, :, :]

In [ ]:
Data_SS = {'rabi_SS_sweep_arr': rabi_SS_sweep_arr,
           'rabi_SS_amp': SS_amp,
           'rabi_SS_it': SS_it,
           'port_delay_sw': port_delay_sw,
           'ro_len': ro_len,
           'n_av': n_average,
           'comment': 'Sweep delay and ro len, array dim [delay, ro_len, n_av, 0 or 1]'
            }

Data_SS.update(qubit_parameters)
Data_SS.update(lo_settings)

file_name = 'Single_shots_0_pi_sweep_delay_and_ro_len_'
file_path = get_path_to_file(file_name, '.mat')
savemat(file_path, Data_SS)